# Hugging Face Ready-to-Use Sentiment Analysis

## Objective

We use Hugging Face pretrained pipelines.

We do not train.

We directly use a ready-made sentiment model.

We use China mirror.

We use PyTorch backend.

We test on `imdb_top_500.csv`.

* 1 = Positive
* 0 = Negative

---

## Step 1: Install Libraries

In [1]:
!pip install transformers pandas torch -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


---

## Step 2: Set Hugging Face Mirror

In [2]:
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

---

## Step 3: Import Libraries

In [3]:
import pandas as pd

from transformers import pipeline

---

## Step 4: Load Dataset

In [4]:
df = pd.read_csv("./datasets/imdb_top_500.csv")

In [5]:
print("Dataset size:", len(df))

print("Columns:", df.columns.tolist())

print("\nFirst review preview:")
print(df["text"].iloc[0][:300])

print("\nFirst label:", df["label"].iloc[0])

print("First rating:", df["rating"].iloc[0])

Dataset size: 500
Columns: ['text', 'label', 'rating']

First review preview:
There are lots of extremely good-looking people in this movie. That's probably the best thing about it. Perhaps that even makes it worth watching.<br /><br />"Loaded" tells the story of Tristan Price (Jesse Metcalfe), a young man who's about to make his mark on the world. He's the son of a well-to-d

First label: 0
First rating: 4


---

## Step 5: Build Hugging Face Pipeline

We use a stable pretrained model.

In [6]:
classifier = pipeline(
    task="sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    tokenizer="distilbert-base-uncased-finetuned-sst-2-english",
    framework="pt"
)

/Users/lunde/.pyenv/versions/3.10.6/envs/lewagon/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [7]:
print("Pipeline loaded successfully.")

print("Model name:", classifier.model.name_or_path)

Pipeline loaded successfully.
Model name: distilbert-base-uncased-finetuned-sst-2-english


---

## Step 6: Check Label Style

In [8]:
print("Model labels:")
print(classifier.model.config.id2label)

Model labels:
{0: 'NEGATIVE', 1: 'POSITIVE'}


---

## Step 7: Test One Positive Sentence

In [9]:
result = classifier("This movie was fantastic and exciting")

In [10]:
print("Prediction:")
print(result)

Prediction:
[{'label': 'POSITIVE', 'score': 0.99988853931427}]


---

## Step 8: Test One Negative Sentence

In [11]:
result = classifier("This movie was terrible and boring")

In [12]:
print("Prediction:")
print(result)

Prediction:
[{'label': 'NEGATIVE', 'score': 0.9997912049293518}]


---

## Step 9: Predict First 10 IMDB Reviews

In [13]:
sample_reviews = df["text"].iloc[:10].tolist()

In [14]:
predictions = classifier(
    sample_reviews,
    truncation=True
)

In [15]:
print("Total predictions:", len(predictions))

print("\nFirst prediction:")
print(predictions[0])

Total predictions: 10

First prediction:
{'label': 'NEGATIVE', 'score': 0.9940255880355835}


---

## Step 10: Convert POSITIVE and NEGATIVE into 1 and 0

In [16]:
def hf_to_label(pred):
    return 1 if pred["label"] == "POSITIVE" else 0

In [17]:
converted_preds = [
    hf_to_label(pred)
    for pred in predictions
]

In [18]:
print("Converted predictions:")
print(converted_preds)

Converted predictions:
[0, 1, 0, 0, 0, 0, 0, 0, 1, 1]


---

## Step 11: Compare with Real Labels

In [19]:
true_labels = df["label"].iloc[:10].tolist()

In [20]:
for i in range(10):
    print(f"\nReview {i+1}")
    print("-" * 50)

    print(df["text"].iloc[i][:200])

    print("\nTrue Label:", true_labels[i])

    print("Predicted Label:", converted_preds[i])

    if true_labels[i] == converted_preds[i]:
        print("Result: Correct")
    else:
        print("Result: Incorrect")


Review 1
--------------------------------------------------
There are lots of extremely good-looking people in this movie. That's probably the best thing about it. Perhaps that even makes it worth watching.<br /><br />"Loaded" tells the story of Tristan Price 

True Label: 0
Predicted Label: 0
Result: Correct

Review 2
--------------------------------------------------
Home Room deals with a Columbine-like high-school shooting but rather than hashing over the occurrence itself the film portrays the aftermath and what happened to the survivors, their trauma, guilt an

True Label: 1
Predicted Label: 1
Result: Correct

Review 3
--------------------------------------------------
If I had not read Pat Barker's 'Union Street' before seeing this film, I would have liked it. Unfortuntately this is not the case. It is actually my kind of film, it is well made, and in no way do I w

True Label: 0
Predicted Label: 0
Result: Correct

Review 4
--------------------------------------------------
"Co

---

## Step 12: Run on First 50 Reviews

In [21]:
subset_size = 50

reviews = df["text"].iloc[:subset_size].tolist()

true_labels = df["label"].iloc[:subset_size].tolist()

In [22]:
predictions = classifier(
    reviews,
    truncation=True,
    batch_size=8
)

In [23]:
pred_labels = [
    hf_to_label(pred)
    for pred in predictions
]

In [24]:
print("Finished predicting", len(pred_labels), "reviews.")

Finished predicting 50 reviews.


---

## Step 13: Compute Accuracy

In [25]:
correct = sum(
    p == y
    for p, y in zip(pred_labels, true_labels)
)

accuracy = correct / len(true_labels)

In [26]:
print("Accuracy on first", subset_size, "reviews:", accuracy)

Accuracy on first 50 reviews: 0.88


---

## Step 14: View Confidence Scores

In [27]:
for i in range(5):
    print(f"\nReview {i+1}")

    print("HF Label:", predictions[i]["label"])

    print("Confidence Score:", predictions[i]["score"])


Review 1
HF Label: NEGATIVE
Confidence Score: 0.9940255880355835

Review 2
HF Label: POSITIVE
Confidence Score: 0.9937600493431091

Review 3
HF Label: NEGATIVE
Confidence Score: 0.9467043280601501

Review 4
HF Label: NEGATIVE
Confidence Score: 0.963310182094574

Review 5
HF Label: NEGATIVE
Confidence Score: 0.9695209860801697


---

## Step 15: Compare Real Reviews More Clearly

In [28]:
for i in range(5):
    print(f"\nReview {i+1}")
    print("-" * 60)

    print(reviews[i][:300])

    print("\nTrue Label:", true_labels[i])

    print("Predicted Label:", pred_labels[i])

    if true_labels[i] == pred_labels[i]:
        print("Result: Correct")
    else:
        print("Result: Incorrect")


Review 1
------------------------------------------------------------
There are lots of extremely good-looking people in this movie. That's probably the best thing about it. Perhaps that even makes it worth watching.<br /><br />"Loaded" tells the story of Tristan Price (Jesse Metcalfe), a young man who's about to make his mark on the world. He's the son of a well-to-d

True Label: 0
Predicted Label: 0
Result: Correct

Review 2
------------------------------------------------------------
Home Room deals with a Columbine-like high-school shooting but rather than hashing over the occurrence itself the film portrays the aftermath and what happened to the survivors, their trauma, guilt and denial.<br /><br />*Spoilers* The shooting itself is treated as a foregone conclusion, with no act

True Label: 1
Predicted Label: 1
Result: Correct

Review 3
------------------------------------------------------------
If I had not read Pat Barker's 'Union Street' before seeing this film, I would have l

---

## Step 16: Predict Custom Reviews

In [29]:
custom_reviews = [
    "This movie was amazing with brilliant acting",
    "I hated this film so much",
    "It was okay, not bad, not great"
]

In [30]:
custom_preds = classifier(
    custom_reviews,
    truncation=True
)

In [31]:
for review, pred in zip(custom_reviews, custom_preds):
    print("\nReview:")
    print(review)

    print("HF Label:", pred["label"])

    print("Numeric Label:", hf_to_label(pred))

    print("Score:", pred["score"])


Review:
This movie was amazing with brilliant acting
HF Label: POSITIVE
Numeric Label: 1
Score: 0.9998835325241089

Review:
I hated this film so much
HF Label: NEGATIVE
Numeric Label: 0
Score: 0.9996534585952759

Review:
It was okay, not bad, not great
HF Label: NEGATIVE
Numeric Label: 0
Score: 0.7728215456008911


---

## Step 17: Simple Batch Prediction Function

In [32]:
def predict_sentiment(text_list):
    preds = classifier(
        text_list,
        truncation=True
    )

    return [
        {
            "text": text,
            "hf_label": pred["label"],
            "numeric_label": hf_to_label(pred),
            "score": pred["score"]
        }
        for text, pred in zip(text_list, preds)
    ]

In [33]:
sample_results = predict_sentiment([
    "What a wonderful movie",
    "This was a disaster"
])

for item in sample_results:
    print("\nText:", item["text"])

    print("HF Label:", item["hf_label"])

    print("Numeric Label:", item["numeric_label"])

    print("Score:", item["score"])


Text: What a wonderful movie
HF Label: POSITIVE
Numeric Label: 1
Score: 0.9998807907104492

Text: This was a disaster
HF Label: NEGATIVE
Numeric Label: 0
Score: 0.9997826218605042
